**Using Zero-Shot Classification to categorize books into four broad genres based on their descriptions.**


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('df_cleaned.csv')

In [3]:
categories = df['categories'].value_counts().reset_index().query('count >= 50')['categories']

In [4]:
categories.tolist()

['Fiction',
 'Juvenile Fiction',
 'Biography & Autobiography',
 'History',
 'Literary Criticism',
 'Religion',
 'Philosophy',
 'Comics & Graphic Novels',
 'Drama',
 'Juvenile Nonfiction',
 'Science',
 'Poetry',
 'Literary Collections']

In [5]:
len(df['categories'].unique())

480

In [6]:
category_mapping = {'Fiction':"Fiction",
 'Juvenile Fiction':"Children Fiction",
 'Biography & Autobiography':"Nonfiction",
 'History':"Nonfiction",
 'Literary Criticism':"Nonfiction",
 'Religion':"Nonfiction",
 'Philosophy':"Nonfiction",
 'Comics & Graphic Novels':"Fiction",
 'Drama':"Fiction",
 'Juvenile Nonfiction':"Children Nonfiction",
 'Science':"Nonfiction",
 'Poetry':"Fiction"}

In [7]:
df['simple_category'] = df['categories'].map(category_mapping)

In [8]:
df['simple_category']

0          Fiction
1              NaN
2          Fiction
3              NaN
4              NaN
           ...    
5192           NaN
5193           NaN
5194           NaN
5195    Nonfiction
5196    Nonfiction
Name: simple_category, Length: 5197, dtype: object

In [9]:
len(df[~df['simple_category'].isna()]),len(df[df['simple_category'].isna()])

(3743, 1454)

**Model Choice: Facebook's BART-Large-MNLI**

In [10]:
from transformers import pipeline

In [11]:
categories = ['Fiction', 'Nonfiction']

In [12]:
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli', device=0)

Device set to use cuda:0


In [13]:
sequence = df.loc[df['simple_category'] == 'Fiction', 'description'].reset_index(drop=True)[0]

In [14]:
# Running through the classifier to predict the category
classifier(sequence, categories)

{'sequence': 'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst

In [15]:
# Getting the max label
max_index = np.argmax(classifier(sequence, categories)['scores'])
max_label = classifier(sequence, categories)['labels'][max_index]
max_label

'Fiction'

**Creating a function to generate predictions.**

In [16]:
def generate_predictions(sequence, categories):
    predictions = classifier(sequence, categories)
    max_index = np.argmax(predictions['scores'])
    max_label = predictions['labels'][max_index]
    return max_label

In [17]:
generate_predictions('it a book about the legend of assassins creed', categories)

'Fiction'

In [18]:
from tqdm import tqdm

actual_cats = []
predicted_cats = []

for i in tqdm(range(0, 300)):
    sequence = df.loc[df['simple_category'] == 'Fiction', 'description'].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, categories)]
    actual_cats += ['Fiction']

100%|██████████| 300/300 [00:41<00:00,  7.14it/s]


In [19]:
for i in tqdm(range(0, 300)):
    sequence = df.loc[df['simple_category'] == 'Nonfiction', 'description'].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, categories)]
    actual_cats += ['Nonfiction']

100%|██████████| 300/300 [00:42<00:00,  7.06it/s]


In [20]:
predictions_df = pd.DataFrame({'actual_categories': actual_cats, 'predicted_categories': predicted_cats})

In [21]:
predictions_df

,actual_categories,predicted_categories
0,Fiction,Fiction
1,Fiction,Fiction
2,Fiction,Fiction
3,Fiction,Nonfiction
4,Fiction,Fiction
...,...,...
595,Nonfiction,Nonfiction
596,Nonfiction,Fiction
597,Nonfiction,Nonfiction
598,Nonfiction,Nonfiction


In [22]:
# Calculating accuracy
predictions_df['correct_prediction'] = (
    np.where(predictions_df['actual_categories'] == predictions_df['predicted_categories'], 1, 0) 
)

In [23]:
predictions_df['correct_prediction'].sum() / len(predictions_df)

np.float64(0.7783333333333333)

In [24]:
# Predicting missing categories
isbns = []
predicted_cats = []

missing_cats = df.loc[df['simple_category'].isna(), ['isbn13', 'description']].reset_index(drop=True)

In [25]:
# Running predictions on missing categories
for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats['description'][i]
    predicted_cats += [generate_predictions(sequence, categories)]
    isbns += [missing_cats['isbn13'][i]]

100%|██████████| 1454/1454 [03:20<00:00,  7.26it/s]


In [26]:
# Creating dataframe of predicted categories for missing data
missing_predicted_df = pd.DataFrame({'isbn13': isbns, 'predicted_category': predicted_cats})
missing_predicted_df

,isbn13,predicted_category
0,9780002261982,Fiction
1,9780006280897,Nonfiction
2,9780006280934,Nonfiction
3,9780006380832,Nonfiction
4,9780006470229,Fiction
...,...,...
1449,9788125026600,Nonfiction
1450,9788171565641,Fiction
1451,9788172235222,Fiction
1452,9788173031014,Nonfiction


In [27]:
# Merging predicted categories back into original dataframe
df = pd.merge(df, missing_predicted_df, on='isbn13', how='left')
df['simple_category'] = np.where(df['simple_category'].isna(), df['predicted_category'], df['simple_category'])
df = df.drop(columns=['predicted_category'])

In [28]:
df

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title and subtitle,tagged_description,simple_category
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Fiction
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction


In [29]:
# Filtering to specific categories 
df[df['categories'].str.lower().isin([
    'romance',
    'science fiction',
    'fantasy',
    'horror',
    'mystery',
    'thriller',
    'comedy'
    'crime,'
    'historical fiction'
])]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title and subtitle,tagged_description,simple_category
24,9780006513087,0006513085,Gravity,Tess Gerritsen,Science fiction,http://books.google.com/books/content?id=KI66c...,Emma Watson a research physician has been trai...,2004.0,4.04,342.0,8024.0,Gravity,9780006513087 Emma Watson a research physician...,Nonfiction
475,9780099410355,0099410354,Traitor,Matthew Woodring Stover,Science fiction,http://books.google.com/books/content?id=VbICO...,"From the depths of catastrophe, a glimmer of h...",2002.0,4.00,320.0,6765.0,Traitor,"9780099410355 From the depths of catastrophe, ...",Fiction
491,9780099446729,0099446723,Blackwood Farm,Anne Rice,Horror,http://books.google.com/books/content?id=cIn8T...,"Lestat Is Back, Saviour And Demon, Presiding O...",2003.0,3.86,774.0,26145.0,Blackwood Farm,"9780099446729 Lestat Is Back, Saviour And Demo...",Fiction
1090,9780261102422,0261102427,The Silmarillion,John Ronald Reuel Tolkien,Fantasy,http://books.google.com/books/content?id=22ePu...,Tolkien's Silmarillion is the core work of the...,1999.0,3.91,384.0,253.0,The Silmarillion,9780261102422 Tolkien's Silmarillion is the co...,Fiction
1435,9780340837955,0340837950,Stranger in a Strange Land,Robert A. Heinlein,Science fiction,http://books.google.com/books/content?id=ZQhiP...,"Epic, entertaining, Stranger in a Strange Land...",2005.0,3.92,672.0,563.0,Stranger in a Strange Land,"9780340837955 Epic, entertaining, Stranger in ...",Fiction
1439,9780345251220,0345251229,Visions from Nowhere,William Arrow,Science fiction,NaN,"The first novel in the series, ""Return to the ...",1976.0,3.23,183.0,10.0,Visions from Nowhere,"9780345251220 The first novel in the series, ""...",Fiction
2845,9780575075597,0575075597,Replay,Ken Grimwood,Fantasy,http://books.google.com/books/content?id=9vmNP...,At forty-three Jeff Winston is tired of his lo...,2005.0,4.16,272.0,412.0,Replay,9780575075597 At forty-three Jeff Winston is t...,Fiction
2860,9780590254762,0590254766,"The lion, the witch and the wardrobe",Clive Staples Lewis,Fantasy,NaN,Four English school children enter the magic l...,1995.0,4.21,189.0,860.0,"The lion, the witch and the wardrobe",9780590254762 Four English school children ent...,Nonfiction
3288,9780739423851,0739423851,Wizard's Castle,Diana Wynne Jones,Fantasy,http://books.google.com/books/content?id=hB7hA...,Howl's moving castle - Eldest of three sisters...,2002.0,4.44,376.0,439.0,Wizard's Castle,9780739423851 Howl's moving castle - Eldest of...,Fiction
3289,9780739439708,0739439707,Time Quartet,Madeleine L'Engle,Science fiction,NaN,"Blending magic with quantum physics, Madeleine...",2003.0,4.35,646.0,165.0,Time Quartet,9780739439708 Blending magic with quantum phys...,Fiction


In [30]:
df.to_csv('df_with_categories.csv', index=False)